# SMS Spam Collection — EDA and Model Evaluation

## Team
- Edwin Gómez
- Johan Sebastián Bonilla

## Goal
This notebook performs:
1. **Exploratory Data Analysis (EDA)** on the **SMS Spam Collection** dataset.
2. **Evaluation of the Hugging Face model** `Goodmotion/spam-mail-classifier` on a **random sample of 20 SMS messages**.

> **Important note:** the selected model is a generic spam classifier hosted on Hugging Face. Since it was not necessarily trained specifically on the SMS Spam Collection dataset, this evaluation should be considered **exploratory**.


## Dataset files used in this notebook
- Dataset file: `/mnt/data/SMSSpamCollection`
- Readme file: `/mnt/data/readme`

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from collections import Counter
import re
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report
from transformers import pipeline

plt.rcParams["figure.figsize"] = (10, 5)
pd.set_option("display.max_colwidth", 200)

## 1. Load dataset

In [ ]:
dataset_path = r"/mnt/data/SMSSpamCollection"
df = pd.read_csv(dataset_path, sep="\t", header=None, names=["label", "message"])
df.head()

In [ ]:
print("Shape:", df.shape)
print("\nClass distribution:")
print(df["label"].value_counts())
print("\nMissing values:")
print(df.isna().sum())

## 2. Basic preprocessing and feature engineering

In [ ]:
df["message"] = df["message"].astype(str)
df["char_length"] = df["message"].str.len()
df["word_count"] = df["message"].str.split().str.len()
df["digit_count"] = df["message"].str.count(r"\d")
df["uppercase_count"] = df["message"].apply(lambda x: sum(1 for c in x if c.isupper()))
df["exclamation_count"] = df["message"].str.count("!")
df["has_url"] = df["message"].str.contains(r"http|www\.|\.com|\.co|\.uk", case=False, regex=True)

df.head()

## 3. Exploratory Data Analysis (EDA)

### 3.1 Class distribution

In [ ]:
class_counts = df["label"].value_counts()
display(class_counts)

ax = class_counts.plot(kind="bar", title="Class Distribution")
ax.set_xlabel("Label")
ax.set_ylabel("Count")
plt.show()

### 3.2 Character length by class

In [ ]:
df.groupby("label")["char_length"].describe()

In [ ]:
for label in df["label"].unique():
    df[df["label"] == label]["char_length"].plot(kind="hist", alpha=0.6, bins=30, label=label)
plt.title("Message Character Length Distribution")
plt.xlabel("Characters")
plt.ylabel("Frequency")
plt.legend()
plt.show()

### 3.3 Word count by class

In [ ]:
df.groupby("label")["word_count"].describe()

In [ ]:
for label in df["label"].unique():
    df[df["label"] == label]["word_count"].plot(kind="hist", alpha=0.6, bins=30, label=label)
plt.title("Message Word Count Distribution")
plt.xlabel("Words")
plt.ylabel("Frequency")
plt.legend()
plt.show()

### 3.4 Additional text features

In [ ]:
feature_summary = df.groupby("label")[[
    "digit_count", "uppercase_count", "exclamation_count", "has_url"
]].mean()

feature_summary

Interpretation hints:
- **digit_count**: spam often contains prices, phone numbers, or codes.
- **uppercase_count**: spam messages frequently use uppercase words to attract attention.
- **exclamation_count**: spam may rely on urgency and exaggerated punctuation.
- **has_url**: spam frequently contains links or website references.


### 3.5 Most common words in ham vs spam

In [ ]:
STOPWORDS = {
    "the","a","an","and","or","to","of","in","on","for","is","it","i","im","i'm","you","u",
    "me","my","we","our","your","be","are","am","at","this","that","with","so","but","if",
    "was","were","have","has","had","do","did","will","would","can","could","not","from",
    "as","by","he","she","they","them","his","her","their","us"
}

def tokenize(text):
    text = text.lower()
    tokens = re.findall(r"[a-zA-Z']+", text)
    return [t for t in tokens if t not in STOPWORDS and len(t) > 1]

ham_tokens = []
spam_tokens = []

for _, row in df.iterrows():
    toks = tokenize(row["message"])
    if row["label"] == "ham":
        ham_tokens.extend(toks)
    else:
        spam_tokens.extend(toks)

ham_top = Counter(ham_tokens).most_common(20)
spam_top = Counter(spam_tokens).most_common(20)

ham_top_df = pd.DataFrame(ham_top, columns=["word", "count"])
spam_top_df = pd.DataFrame(spam_top, columns=["word", "count"])

print("Top words in HAM:")
display(ham_top_df)

print("Top words in SPAM:")
display(spam_top_df)

In [ ]:
plt.figure(figsize=(12,5))
plt.bar(ham_top_df["word"], ham_top_df["count"])
plt.xticks(rotation=70)
plt.title("Top 20 HAM Words")
plt.show()

plt.figure(figsize=(12,5))
plt.bar(spam_top_df["word"], spam_top_df["count"])
plt.xticks(rotation=70)
plt.title("Top 20 SPAM Words")
plt.show()

## 4. EDA conclusions

Typical patterns often observed in this dataset:
- **Ham** messages are much more frequent than spam messages.
- **Spam** messages tend to be longer and may contain more promotional language.
- Spam messages often include:
  - numbers
  - URLs / references to websites
  - uppercase words
  - exclamation marks
- Frequent spam words usually include terms related to:
  - prizes
  - free offers
  - claim / call / text
  - cash or rewards


## 5. Load Hugging Face model

In [ ]:
classifier = pipeline(
    "text-classification",
    model="Goodmotion/spam-mail-classifier"
)
print("Model loaded successfully.")

## 6. Random sample of 20 messages for testing

For each sampled message we will show:
- the **true label**
- the **predicted label**
- the **confidence score** returned by the model
- whether the prediction was correct


In [ ]:
sample_df = df.sample(20, random_state=42).copy()
sample_df = sample_df.reset_index(drop=True)
sample_df[["label", "message"]].head()

In [ ]:
def normalize_prediction(raw_label: str) -> str:
    label = str(raw_label).lower()
    if "spam" in label:
        return "spam"
    if "ham" in label:
        return "ham"
    # fallback if the model uses generic labels such as LABEL_0 / LABEL_1
    return label

predicted_labels = []
scores = []
correct_flags = []

for msg, true_label in zip(sample_df["message"], sample_df["label"]):
    result = classifier(msg)[0]
    pred_label = normalize_prediction(result["label"])
    score = float(result["score"])

    predicted_labels.append(pred_label)
    scores.append(score)
    correct_flags.append(pred_label == true_label)

sample_df["predicted_label"] = predicted_labels
sample_df["confidence_score"] = scores
sample_df["correct"] = correct_flags

sample_df[["label", "predicted_label", "confidence_score", "correct", "message"]]

### About “precision of each result”
In this notebook, the per-message value reported is the **confidence score** returned by the model for the predicted class.

A formal **precision metric** is computed over the whole sample, not for a single row. Therefore:
- **confidence_score** = per-message confidence of the prediction
- **precision** = global metric computed after evaluating all 20 samples


## 7. Evaluation metrics on the 20-message sample

In [ ]:
y_true = sample_df["label"].tolist()
y_pred = sample_df["predicted_label"].tolist()

accuracy = accuracy_score(y_true, y_pred)
precision_spam = precision_score(y_true, y_pred, pos_label="spam", zero_division=0)
recall_spam = recall_score(y_true, y_pred, pos_label="spam", zero_division=0)
f1_spam = f1_score(y_true, y_pred, pos_label="spam", zero_division=0)

metrics_df = pd.DataFrame({
    "metric": ["accuracy", "precision_spam", "recall_spam", "f1_spam"],
    "value": [accuracy, precision_spam, recall_spam, f1_spam]
})
metrics_df

In [ ]:
print(classification_report(y_true, y_pred, zero_division=0))

## 8. Final remarks

### Main takeaways
1. The **SMS Spam Collection** dataset is highly useful for binary text classification experiments.
2. Spam messages usually show stronger lexical signals such as promotional vocabulary, urgency, numbers, and links.
3. The Hugging Face model can be used as a practical inference layer for quick testing, but the results should be interpreted carefully because the model was not necessarily trained on this exact dataset.

### Limitations
- Only **20 random messages** were used for the evaluation requested in this assignment.
- The model may have **domain mismatch** with SMS-specific language.
- The first execution may require downloading model weights from Hugging Face.

### Suggested next steps
- Evaluate the model on the **full dataset**.
- Build a **confusion matrix** over a larger test set.
- Compare the Hugging Face model against a custom baseline such as:
  - TF-IDF + Logistic Regression
  - TF-IDF + Naive Bayes
  - DistilBERT fine-tuned on SMS Spam Collection
